In [2]:
import numpy as np
import pandas as pd
from joblib import dump, load
import os
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    r2_score,
    mean_absolute_error, 
    mean_squared_error,
    root_mean_squared_error, 
    mean_absolute_percentage_error,
    root_mean_squared_log_error,
    make_scorer
)
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical, Real

### Data importation

In [4]:
df = pd.read_csv("../data/processed/bc_clean.csv")
df = df.drop(["Unnamed: 0"], axis=1)
df.head(3)

,latitude,longitude,price,property-beds,property-baths,Acreage,Property Tax,Square Footage,Missing Acreage,Missing Property Tax,...,heat_pump,overhead,space_heater,Property Type_Condo,Property Type_Condo/Townhome,Property Type_Duplex,Property Type_Manufactured Home,Property Type_MultiFamily,Property Type_Single Family,Property Type_Townhome
0,49.821860,-119.480143,1298000.0,5.0,4.0,0.69,6995.0,4374.0,0,0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,49.138904,-122.654191,1399999.0,6.0,4.0,0.04,2585.0,2404.0,0,0,...,0,0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,49.103726,-122.663125,399900.0,1.0,1.0,0.00,1474.0,632.0,1,0,...,0,0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
X = df.drop(["price"], axis=1)
Y = df["price"]

#### train test split

In [6]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42, shuffle=True
)

#### BayesSearchCV

In [70]:
search_space = {
    "n_estimators" : Integer(100, 600, prior="log-uniform"),
    "max_depth": Categorical([None] + list(range(2, 51))),
    "min_samples_split" : Integer(2, 100, prior="log-uniform"),
    "min_samples_leaf" : Integer(1, 50, prior="log-uniform"),
    "bootstrap" : Categorical([True, False])    
}

scoring = {
    "r2_score" : make_scorer(r2_score, greater_is_better=True),
    "neg_mae" : make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_mse" : make_scorer(mean_squared_error, greater_is_better=False),
    "neg_rmse" : make_scorer(root_mean_squared_error, greater_is_better=False),
    "neg_rmsle" : make_scorer(root_mean_squared_log_error, greater_is_better=False),
    "neg_mape" : make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}

In [75]:
rdf_search = BayesSearchCV(
    estimator = RandomForestRegressor(max_features="sqrt", n_jobs=1, random_state=42),
    search_spaces=search_space,
    scoring = scoring["neg_rmsle"],  # It penalizes errors for cheap houses and less for expensive houses
    n_iter = 50,
    cv=KFold(3),
    n_jobs=3,
    error_score="raise",
    verbose=2
)

In [77]:
if os.path.isfile("../artifacts/rdf_search.pkl"):
    print("la grille existe déjà")
else : 
    rdf_search.fit(X_train, Y_train)
    dump(rdf_search, "../artifacts/rdf_search.pkl")
    print("L'objet a été sauvegardé")

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits


/home/marwane/canada-housing-predictor/venv/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.False_, np.int64(24), np.int64(1), np.int64(2), np.int64(100)] before, using random point [False, 30, np.int64(1), np.int64(14), np.int64(198)]
  warnings.warn(


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits


/home/marwane/canada-housing-predictor/venv/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.False_, np.int64(29), np.int64(1), np.int64(2), np.int64(600)] before, using random point [True, 35, np.int64(13), np.int64(2), np.int64(142)]
  warnings.warn(


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
[CV] END bootstrap=False, max_depth=34, min_samples_leaf=34, min_samples_split=4, n_estimators=195; total time=   1.9s
[CV] END bootstrap=False, max_depth=10, min_samples_leaf=11, min_samples_split=24, n_estimators=215; total time=   2.0s
[CV] END bootstrap=True, max_depth=12, min_samples_leaf=4, min_samples_split=23, n_estimators=161; total time=   1.2s
[CV] END bootstrap=True, max_depth=40, min_samples_leaf=39, min_samples_split=63, n_estimators=362; total time=   2.3s
[CV] END bootstrap=True, max_depth=9, min_samples_leaf=27, min_samples_split=6, n_estimators=104; total time=   0.7s
[CV] END bootstrap=True, max_depth=24, min_samples_leaf=19, min_samples_split=8, n_estimators=259; total time=   2.0s
[CV] END bootstrap=True, max_depth=33, min_samples_leaf=2, min_samples_split=4, n_estimators=294; total time=   

...

Fitting 3 folds for each of 1 candidates, totalling 3 fits


/home/marwane/canada-housing-predictor/venv/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.False_, np.int64(24), np.int64(1), np.int64(2), np.int64(600)] before, using random point [True, 22, np.int64(45), np.int64(43), np.int64(102)]
  warnings.warn(


Fitting 3 folds for each of 1 candidates, totalling 3 fits


/home/marwane/canada-housing-predictor/venv/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.False_, None, np.int64(1), np.int64(2), np.int64(600)] before, using random point [True, 50, np.int64(6), np.int64(25), np.int64(163)]
  warnings.warn(


Fitting 3 folds for each of 1 candidates, totalling 3 fits
L'objet a été sauvegardé


In [ ]:
rdf_search = load("../artifacts/rdf_search.pkl")
rdf_search.best_params_

In [86]:
best_model_dtree = rdf_search.best_estimator_

y_test_pred = best_model_dtree.predict(X_test)
y_train_pred = best_model_dtree.predict(X_train)

In [87]:
print("R²:", r2_score(Y_test, y_test_pred))
print("R²:", r2_score(Y_train, y_train_pred))

print("MAE:", mean_absolute_error(Y_test, y_test_pred))
print("MAE:", mean_absolute_error(Y_train, y_train_pred))

print("RMSE:", root_mean_squared_error(Y_test, y_test_pred))
print("RMSE:", root_mean_squared_error(Y_train, y_train_pred))

print("MAPE:", mean_absolute_percentage_error(Y_test, y_test_pred))
print("MAPE:", mean_absolute_percentage_error(Y_train, y_train_pred))

print("RMSLE", root_mean_squared_log_error(Y_test, y_test_pred))
print("RMSLE", root_mean_squared_log_error(Y_train, y_train_pred))

R²: 0.8023851752800182
R²: 0.9999159687072146
MAE: 288459.3030090133
MAE: 3373.1697010407283
RMSE: 769433.1079430275
RMSE: 16815.351223621397
MAPE: 0.17440104918230906
MAPE: 0.002876866170290809
RMSLE 0.2361103990278528
RMSLE 0.009562733711026734


The random forest is good but overfitting. We can clearly see a gap between the train set and the test set, between all the scores.
I'll re-train the random forest with different hyperparameters.  
The overfitting is due to deep depth of each tree and the min samples leaf/split that are too small. All these values makes the trees specialized to the train set.

#### Search n°2

In [94]:
search_space2 = {
    "max_depth": Integer(3, 15),
    "min_samples_split" : Integer(30, 100),
    "min_samples_leaf" : Integer(20, 100),
    "max_samples" : Real(0.5, 0.9)
}

rdf_search2 = BayesSearchCV(
    estimator = RandomForestRegressor(n_estimators=500,bootstrap=True, max_features="sqrt", n_jobs=1, random_state=42),
    search_spaces=search_space2,
    scoring = make_scorer(root_mean_squared_log_error, greater_is_better=False),  # It penalizes errors for cheap houses and less for expensive houses
    n_iter = 50,
    cv=KFold(3),
    n_jobs=3,
    error_score="raise",
    verbose=2,
    random_state=42
)

In [96]:
if os.path.isfile("../artifacts/rdf_search2.pkl"):
    print("la grille existe déjà")
else : 
    rdf_search2.fit(X_train, Y_train)
    dump(rdf_search2, "../artifacts/rdf_search2.pkl")
    print("L'objet a été sauvegardé")

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fi

In [97]:
rdf_search2 = load("../artifacts/rdf_search2.pkl")
rdf_search2.best_params_

OrderedDict([('max_depth', 15),
             ('max_samples', 0.9),
             ('min_samples_leaf', 20),
             ('min_samples_split', 30)])

In [99]:
best_model_dtree2 = rdf_search2.best_estimator_

y_test_pred2 = best_model_dtree2.predict(X_test)
y_train_pred2 = best_model_dtree2.predict(X_train)

In [101]:
print("R²:", r2_score(Y_test, y_test_pred2))
print("R²:", r2_score(Y_train, y_train_pred2))

print("MAE:", mean_absolute_error(Y_test, y_test_pred2))
print("MAE:", mean_absolute_error(Y_train, y_train_pred2))

print("RMSE:", root_mean_squared_error(Y_test, y_test_pred2))
print("RMSE:", root_mean_squared_error(Y_train, y_train_pred2))

print("MAPE:", mean_absolute_percentage_error(Y_test, y_test_pred2))
print("MAPE:", mean_absolute_percentage_error(Y_train, y_train_pred2))

print("RMSLE", root_mean_squared_log_error(Y_test, y_test_pred2))
print("RMSLE", root_mean_squared_log_error(Y_train, y_train_pred2))

R²: 0.7313729178520456
R²: 0.7044669036982871
MAE: 371740.74071823526
MAE: 367272.3625253878
RMSE: 897089.9808177118
RMSE: 997214.2648650698
MAPE: 0.2565170954467974
MAPE: 0.25258593040188937
RMSLE 0.30848199202657933
RMSLE 0.30775993544495744


We are no longer overfitting. The scores between the train set and the test set are almost identical.

Let's analyse the errors of the model by groups of prices, in order £to see the errors for each price range. 

In [119]:
df_error = pd.DataFrame({
    "y_true": Y_test,
    "y_pred": y_test_pred2
})

#this creates a new column with "q" different price ranges. We can know where to situtate each row.
df_error["price_bin"] = pd.qcut(df_error["y_true"], q=5)  

# We calculate the Mean Average Percentage Error for each price range.
df_error.groupby("price_bin").apply(
    lambda x: np.mean(np.abs((x.y_true - x.y_pred) / x.y_true))
)

/tmp/ipykernel_39813/512691629.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_error.groupby("price_bin").apply(
/tmp/ipykernel_39813/512691629.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_error.groupby("price_bin").apply(


price_bin
(49499.999, 629900.0]      0.473830
(629900.0, 888480.0]       0.182359
(888480.0, 1300000.0]      0.205791
(1300000.0, 2000000.0]     0.208065
(2000000.0, 29998000.0]    0.212236
dtype: float64

Our model makes an error of 47% for properties that have a price prediction between 49499\\$ and 629900\\$

#### Search n°3

#### This time, we'll transform the Y vector target to a log-Y target vector in order to reduce the errors for the smallest prices of our dataset.  

Indeed, by applying the logarithm to the `Price` feature, the model penalizes much more the errors for the smallest price values than the biggest ones.  
We do that in order to give more importance to the errors on the smallest prices, by removing the dominance of large prices. We change the "space" of the error space thanks to the form of the logartihm function that becomes flatter for large values.

In [121]:
Y_log = np.log1p(Y)
Y_log

0        14.076336
1        14.151983
2        12.898972
3        13.663525
4        13.321216
           ...    
22518    16.801193
22519    14.126666
22520    14.077875
22521    14.206877
22522    13.652993
Name: price, Length: 22523, dtype: float64

In [126]:
X_train2, X_test2, Y_train2, Y_test2 = train_test_split(
    X, Y_log, test_size=0.20, random_state=42, shuffle=True
)

In [127]:
search_space2 = {
    "max_depth": Integer(3, 15),
    "min_samples_split" : Integer(30, 100),
    "min_samples_leaf" : Integer(20, 100),
    "max_samples" : Real(0.5, 0.9)
}

rdf_search3 = BayesSearchCV(
    estimator = RandomForestRegressor(n_estimators=500,bootstrap=True, max_features="sqrt", n_jobs=1, random_state=42),
    search_spaces=search_space2,
    scoring = make_scorer(root_mean_squared_log_error, greater_is_better=False),  # It penalizes errors for cheap houses and less for expensive houses
    n_iter = 50,
    cv=KFold(3),
    n_jobs=3,
    error_score="raise",
    verbose=2,
    random_state=42
)

In [128]:
rdf_search3.fit(X_train2, Y_train2)

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fi

,estimator,RandomForestR...ndom_state=42)
,search_spaces,"{'max_depth': Integer(low=3...m='normalize'), 'max_samples': Real(low=0.5,...m='normalize'), 'min_samples_leaf': Integer(low=2...m='normalize'), 'min_samples_split': Integer(low=3...m='normalize')}"
,optimizer_kwargs,None
,n_iter,50
,scoring,make_scorer(r...hod='predict')
,fit_params,None
,n_jobs,3
,n_points,1
,iid,'deprecated'
,refit,True
,cv,KFold(n_split...shuffle=False)


In [131]:
if os.path.isfile("../artifacts/rdf_search3.pkl"):
    print("la grille existe déjà")
else : 
    rdf_search3.fit(X_train, Y_train)
    dump(rdf_search3, "../artifacts/rdf_search3.pkl")
    print("L'objet a été sauvegardé")

la grille existe déjà


In [3]:
rdf_search3 = load("../artifacts/rdf_search3.pkl")
rdf_search3.best_params_

OrderedDict([('max_depth', 15),
             ('max_samples', 0.9),
             ('min_samples_leaf', 20),
             ('min_samples_split', 30)])

In [5]:
best_model_dtree3 = rdf_search3.best_estimator_

if os.path.isfile("../artifacts/rdf_best_model.pkl"):
    print("la grille existe déjà")
else : 
    dump(best_model_dtree3, "../artifacts/rdf_best_model.pkl")
    print("L'objet a été sauvegardé")

la grille existe déjà


In [ ]:
y_pred_log = best_model_dtree3.predict(X_test2)
y_test_pred3 = np.expm1(y_pred_log)

y_train_pred_log = best_model_dtree3.predict(X_train2)
y_train_pred3 = np.expm1(y_train_pred_log)

In [142]:
print("R²:", r2_score(Y_test, y_test_pred3))
print("R²:", r2_score(Y_train, y_train_pred3))

print("MAE:", mean_absolute_error(Y_test, y_test_pred3))
print("MAE:", mean_absolute_error(Y_train, y_train_pred3))

print("RMSE:", root_mean_squared_error(Y_test, y_test_pred3))
print("RMSE:", root_mean_squared_error(Y_train, y_train_pred3))

print("MAPE:", mean_absolute_percentage_error(Y_test, y_test_pred3))
print("MAPE:", mean_absolute_percentage_error(Y_train, y_train_pred3))

print("RMSLE", root_mean_squared_log_error(Y_test, y_test_pred3))
print("RMSLE", root_mean_squared_log_error(Y_train, y_train_pred3))

R²: 0.6470264928145755
R²: 0.5930937847729365
MAE: 362778.9591370971
MAE: 366574.9416686453
RMSE: 1028329.1904051022
RMSE: 1170125.8725769515
MAPE: 0.20174920748803957
MAPE: 0.1945408483975409
RMSLE 0.2845719117168973
RMSLE 0.28334495059539455


In [143]:
df_error = pd.DataFrame({
    "y_true": Y_test,
    "y_pred": y_test_pred3
})

#this creates a new column with "q" different price ranges. We can know where to situtate each row.
df_error["price_bin"] = pd.qcut(df_error["y_true"], q=5)  

# We calculate the Mean Average Percentage Error for each price range.
df_error.groupby("price_bin").apply(
    lambda x: np.mean(np.abs((x.y_true - x.y_pred) / x.y_true))
)

/tmp/ipykernel_39813/2132851583.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_error.groupby("price_bin").apply(
/tmp/ipykernel_39813/2132851583.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_error.groupby("price_bin").apply(


price_bin
(49499.999, 629900.0]      0.318113
(629900.0, 888480.0]       0.135032
(888480.0, 1300000.0]      0.152161
(1300000.0, 2000000.0]     0.164501
(2000000.0, 29998000.0]    0.238956
dtype: float64

On average, the prediction error magnitude is 31% of the true price for houses with a price prediction that is between 49499\\$ and 629900\\$  
It's less than the previous model (47%). Applying the logarithm to the target vector was a great idea.

#### Conclusion  
We'll keep this model in mind and train Boosting models. Furthermore, instead of giving a unique price prediction with this Random Rorest, we'll give a prediction with intervals of confidence, which correspond to quantiles of the prediction from all the trees of our RandomForest.  
The prediction will be the median value instead of the mean because the median is less sensitive to extreme values / outliers, compared to the mean.